In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
seed = 1987 
import torch  
import numpy as np
torch.manual_seed(seed)
np.random.seed(seed)
from IPython.core.display import HTML, display
import torch.nn as nn
import tiktoken  
from typing import  List

In [3]:
import sys
sys.path.append('../')
import Inference as inf
from src import dataloader
from models.configs import Configs, set_seed
set_seed(1987)
from models.ClimMgmtAwareViTLRP import ClimMgmtAware_ViT
# from models.vit_explaination_generator import LRP

### Text

### Ex003

In [25]:
exp_name = '002_s12mettext_vit_768_8_6_8_30_0001_01_64_init01_attn'
words = inf.TextScoresAnalysis(exp_name= exp_name, batch_size = 64, layer = -1, head = -1, lrp = False).single_visualize_text(block = 162018)

250 250 248
235 235


### Ex000

In [5]:
exp_name = '005_S2S1MT_0001_01_8_6_768_30_64_MSE_Q_Norm_Init_context128'
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 64, layer = -1, head = -1, lrp = False).single_visualize_text(block = 162018)

In [6]:
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 64, layer = -1, head = -1, lrp = False).single_visualize_text(block = 32017)

#### GPT2: 000_S2S1MT_0001_01_8_6_768_30_48_MSE

In [10]:
exp_name = '000_S2S1MT_0001_01_8_6_768_30_48_MSE'
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 48, layer = -1, head = -1, lrp = False).single_visualize_text(block = 162018)

In [11]:
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 48, layer = -1, head = -1, lrp = False).single_visualize_text(block = 32017)

In [12]:
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 48, layer = -1, head = -1, lrp = False).single_visualize_text(block = 72018)

In [10]:
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 48, layer = -1, head = -1, lrp = False).extreme_categ_visualize_text()

#### BERT

In [6]:
exp_name = '002_S2S1MT_0001_01_8_6_768_30_32_MSE_BERT'
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 64, layer = -1, head = -1, lrp = False).single_visualize_text(block = 162018)

In [7]:
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 64, layer = -1, head = -1, lrp = False).single_visualize_text(block = 32017)

In [14]:
exp_name = '002_S2S1MT_0001_01_8_6_768_30_32_MSE_BERT'
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 64, layer = -1, head = -1, lrp = False).extreme_categ_visualize_text()

### Others

In [50]:
class TextAttenVis(nn.Module):
    def __init__(self, tokens: List[int], attns_arr, layer: int, head: int):
        super().__init__()
        self.tokens = tokens
        self.attns_arr = attns_arr  # Assuming attns_arr is a tensor with shape [H, X, Y, L]
        self.layer = layer
        self.head = head
        self.TextEncoder = tiktoken.get_encoding('gpt2')
        
    def decode(self):
        """Decode tokens using the TextEncoder."""
        # Decode both the text and get token offsets
        decoded_text = self.TextEncoder.decode(self.tokens)
        token_bytes = [self.TextEncoder.decode_single_token_bytes(token) for token in self.tokens]
        words = [t.decode('utf-8') for t in token_bytes]
        return words
    
    def calc_attn(self):
        """Calculate the average attention score for a specific layer and head, excluding the first token."""
        # Extract the attention scores for the specified head and layer
        attn_scores = self.attns_arr[self.head, :, :, self.layer]
        # Exclude the first token's attention scores (position 0)
        attn_scores = attn_scores[2:, 2:]
        # Calculate the average across columns
        avg_attn_scores = np.mean(attn_scores, axis=0)

        norm_attn_scores = self.normalize_array(avg_attn_scores)
        return norm_attn_scores
    
    def plot(self):
        """Use decoded text and attention scores to create a visualization."""
        words = self.decode()
        importances = self.calc_attn()

        self.visualize_text(words, importances)

    def normalize_array(self, values):
        min_old = values.min()
        max_old = values.max()
        min_new, max_new =-1, 1
        normalized_values = [(value - min_old) / (max_old - min_old) * (max_new - min_new) + min_new for value in values]
        return np.array(normalized_values, dtype= np.float32)
    
    def format_special_tokens(self, word):
        # Strip underscores often used in tokenized outputs
        return word.replace('_', ' ')


    def _get_color(self, attr):
        # clip values to prevent CSS errors (Values should be from [-1,1])
        attr = max(-1, min(1, attr))
        if attr > 0:
            hue = 120
            sat = 75
            lig = 100 - int(50 * attr)
        else:
            hue = 0
            sat = 75
            lig = 100 - int(-40 * attr)
        return "hsl({}, {}%, {}%)".format(hue, sat, lig)

    def format_word_importances(self, words, importances):
        tags = ["<td>"]
        for word, importance in zip(words, importances):
            color = self._get_color(importance)
            tags.append(
                '<mark style="background-color: {color}; opacity:1.0; line-height:1.75">'
                '<font color="black"> {word} </font></mark>'.format(color=color, word=word)
            )
        tags.append("</td>")
        return "".join(tags)

    def visualize_text(self, words, importances, legend=True):
        assert len(words) == len(importances), "Words and importances must have the same length."
        
        dom = ["<table style='width: 100%;'>"]
        dom.append(
            "<tr>{}</tr>".format(self.format_word_importances(words, importances))
        )
        
        if legend:
            dom.append(
                '<div style="border-top: 1px solid; margin-top: 5px; padding-top: 5px; display: inline-block">'
            )
            dom.append("<b>Legend: </b>")
            for value, label in zip([-1, 0, 1], ["Negative", "Neutral", "Positive"]):
                dom.append(
                    '<span style="display: inline-block; width: 20px; height: 10px; border: 1px solid; background-color: {value};"></span> {label}  '.format(
                        value=self._get_color(value), label=label
                    )
                )
            dom.append("</div>")
        
        dom.append("</table>")
        html = HTML("".join(dom))
        display(html)
        return html

In [5]:
import pandas as pd
exp_df = pd.read_csv('/data2/hkaman/Projects/ViT/EXPs/July/EXP_00_all_lr0001_wd01_dr30_768_8_6_64_yz/00_all_lr0001_wd01_dr30_768_8_6_64_yz_train.csv')
exp_df

,Unnamed: 0,block,cultivar,x,y,ytrue,ypred_w1,ypred_w2,ypred_w3,ypred_w4,...,ypred_w6,ypred_w7,ypred_w8,ypred_w9,ypred_w10,ypred_w11,ypred_w12,ypred_w13,ypred_w14,ypred_w15
0,0,1762018,3,20,0,12.932465,12.296338,12.329173,12.563106,12.336212,...,12.361913,12.381466,12.320471,12.399814,12.465119,12.320650,12.465689,12.417679,12.434526,12.338538
1,1,1762018,3,20,1,13.512397,12.336802,12.398656,12.621766,12.381386,...,12.431398,12.410377,12.365887,12.423085,12.526346,12.348647,12.536717,12.483647,12.498887,12.374754
2,2,1762018,3,20,2,13.879617,12.411592,12.465762,12.706409,12.471702,...,12.473975,12.493038,12.423740,12.504116,12.577646,12.421958,12.589268,12.549308,12.572040,12.465219
3,3,1762018,3,20,3,14.297155,12.407248,12.454989,12.677439,12.428853,...,12.470098,12.469646,12.437248,12.512933,12.576036,12.428190,12.582082,12.554774,12.565578,12.468667
4,4,1762018,3,20,4,13.703508,12.458765,12.508912,12.761071,12.526074,...,12.536178,12.549388,12.501978,12.554325,12.649865,12.485950,12.647227,12.616670,12.639295,12.516236
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3551995,3551995,182017,3,53,25,9.634854,8.471873,8.608835,8.475002,8.566308,...,8.481845,8.625408,8.540093,8.385203,8.752691,8.593314,8.728557,8.637101,8.517480,8.426453
3551996,3551996,182017,3,53,26,9.700796,8.493179,8.671465,8.502865,8.585302,...,8.487825,8.668302,8.543048,8.420088,8.773476,8.590909,8.758548,8.669851,8.572538,8.437601
3551997,3551997,182017,3,53,27,9.907112,8.485063,8.674589,8.517700,8.529644,...,8.474715,8.680202,8.541413,8.352735,8.746447,8.555909,8.744676,8.696676,8.535196,8.414667
3551998,3551998,182017,3,53,28,10.553209,8.446444,8.639217,8.507057,8.565543,...,8.457965,8.646433,8.561075,8.397346,8.706067,8.540813,8.706911,8.657640,8.533182,8.444560


In [6]:
unique_blocks = pd.unique(exp_df['block'])
block_sizes = {}

lower_ = 0
for idx, block in enumerate(unique_blocks):
    size = len(exp_df[exp_df['block'] == block]) / 256
    block_sizes[block] = (lower_, size + lower_)
    lower_ += size

print(block_sizes)

{1762018: (0, 10.0), 1762019: (10.0, 20.0), 162017: (20.0, 74.0), 1022019: (74.0, 300.0), 1032018: (300.0, 817.0), 1112016: (817.0, 858.0), 162019: (858.0, 920.0), 182016: (920.0, 1579.0), 1032019: (1579.0, 2097.0), 762017: (2097.0, 2118.0), 32017: (2118.0, 2503.0), 72016: (2503.0, 2855.0), 32019: (2855.0, 3248.0), 1022017: (3248.0, 3353.0), 162018: (3353.0, 3411.0), 762019: (3411.0, 3900.0), 172016: (3900.0, 3922.0), 122019: (3922.0, 4402.0), 72018: (4402.0, 4735.0), 1072018: (4735.0, 5068.0), 182018: (5068.0, 5727.0), 142017: (5727.0, 5888.0), 382016: (5888.0, 6214.0), 182019: (6214.0, 6893.0), 1032016: (6893.0, 7373.0), 72017: (7373.0, 7598.0), 122016: (7598.0, 8079.0), 682016: (8079.0, 8103.0), 762018: (8103.0, 8591.0), 682017: (8591.0, 8614.0), 142019: (8614.0, 8787.0), 1072017: (8787.0, 9102.0), 1022018: (9102.0, 9311.0), 122018: (9311.0, 9791.0), 142016: (9791.0, 9956.0), 72019: (9956.0, 10291.0), 172018: (10291.0, 10313.0), 1022016: (10313.0, 10536.0), 762016: (10536.0, 11030.0

In [7]:
def process_arrays(directory, range_dict):
    results = {}

    # Iterate over each specified range
    for key, bounds in range_dict.items():
        lower_bound = int(bounds[0])
        upper_bound = int(bounds[1])
        print(lower_bound, upper_bound)
        # Calculate the starting and ending file indices
        start_file_index = lower_bound // 64
        end_file_index = (upper_bound - 1) // 64  # Subtract 1 to handle inclusive upper bound correctly

        # List to store slices of arrays for averaging
        slices = []

        # Loop over the necessary file indices
        for file_index in range(start_file_index, end_file_index + 1):
            filename = f"train_attn_scores_{file_index}.npy"
            file_path = os.path.join(directory, filename)
            
            if os.path.exists(file_path):
                array = np.load(file_path)

                # Calculate slice bounds within the current array
                slice_start = lower_bound - 64 * file_index
                slice_end = upper_bound - 64 * file_index

                # Adjust slice bounds to fit within the current array
                slice_start = max(0, slice_start)
                slice_end = min(64, slice_end)

                if slice_start < slice_end:  # Ensure there is something to slice
                    slices.append(array[slice_start:slice_end])

        # Concatenate all slices along the first axis and compute the mean
        if slices:
            combined_array = np.concatenate(slices, axis=0)
            mean_array = np.percentile(combined_array, 95, axis=0)#np.mean(combined_array, axis=0)
            results[key] = mean_array

    return results


In [ ]:
root_dir = '/data2/hkaman/Projects/ViT/EXPs/July/attnscores'

results = process_arrays(root_dir, block_sizes)
for range_key, mean_array in results.items():
    print(f"Mean for {range_key}: {mean_array.shape}")

In [44]:
arr = np.load('/home/hkaman/Documents/multimodel-transformers-vye/Junky/attn_scores.npy', allow_pickle=True).item()
arr[10]['tokens']

In [46]:
len(arr[10]['tokens'])

248

In [41]:
len(arr[17]['tokens']), results[382017].shape

(245, (8, 250, 250, 6))

In [48]:
_ = TextAttenVis(arr[17]['tokens'], results[1032019], layer = -1, head= -1).plot()

In [51]:
_ = TextAttenVis(arr[10]['tokens'], results[162018], layer = -1, head= -1).plot()

In [23]:
_ = TextAttenVis(text_attns[10]['tokens'], text_attns[10]['text_array'], layer = -1, head= -1).plot()

### LRP

In [13]:
config = Configs(
    img_size = 16, 
    patch_size = 8, 
    embed_dim = 768, 
    context_dim = 768,
    mlp_dim = 512, 
    pool = 'cls',
    in_channels = 8,
    out_channels = 1, 
    num_heads = 8, 
    num_layers = 6, 
    cond = False,
    multi_conv = False,
    attn_dropout = 0.1, 
    proj_dropout = 0.1, 
    drop_path = 0.0,
    post_norm = False, 
    vis = True, 
    tokenizer = 'EC',
    mask_modality = None
    ).call()
import torch
import os
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ClimMgmtAware_ViT(config).to(device)

# exp = '000_s12mettext_vit_768_8_6_8_30_0001_01_32_init01_lrp'
# exp_output_dir = '/data2/hkaman/Projects/ViT/EXPs/Sep' + 'EXP_' + exp
# best_model_name = os.path.join(exp_output_dir, 'best_model_' + exp + '.pth')

model.load_state_dict(torch.load('/data2/hkaman/Projects/ViT/EXPs/Sep/EXP_000_s12mettext_vit_768_8_6_8_30_0001_01_32_init01_lrp/best_model_000_s12mettext_vit_768_8_6_8_30_0001_01_32_init01_lrp.pth'))

<All keys matched successfully>

In [14]:
model.eval()
attribution_generator = LRP(model)

In [15]:
data_loader_training, data_loader_validate, data_loader_test = dataloader.get_dataloaders(
    batch_size = 32, 
    img_size = 16,
    in_channels = 8, 
    resmapling_status = False,
    data = 's2',
    exp_name = 'test'
    )

(13875, 36) | (8233, 36) | (12443, 36)


In [16]:
data_dict_stt = {}
for batch, sample in enumerate(data_loader_training):
    data_dict_stt[sample['block'][0]] = {
    'image': sample['image'][0].to(device),
    'met':sample['met'][0].to(device),
    'text': sample['EmbText'],
    'yz': sample['YZ'][0],
    'mask':sample['mask'][0]
}

In [22]:
for date_time, data in data_dict_stt.items():

    transformer_attribution = attribution_generator.generate_LRP(data['image'].unsqueeze(0).cuda(0), 
                                                                data['text'], 
                                                                 data['met'].unsqueeze(0), 
                                                                 data['yz'].unsqueeze(0),
                                                                 method="transformer_attribution", 
                                                                 index=data['mask']).detach()
    # transformer_attribution = transformer_attribution.reshape(1, 1, 4, 4)
    # transformer_attribution4 = (transformer_attribution - transformer_attribution.min()) / (transformer_attribution.max() - transformer_attribution.min())
    # data_dict_stt[date_time]['lpr'] = transformer_attribution4.data.cpu().numpy()
  
    # transformer_attribution32 = torch.nn.functional.interpolate(transformer_attribution, scale_factor=8, mode='bilinear')
    # transformer_attribution32 = transformer_attribution32.reshape(32, 32)
    # transformer_attribution32 = (transformer_attribution32 - transformer_attribution32.min()) / (transformer_attribution32.max() - transformer_attribution32.min())
    # data_dict_stt[date_time]['lpr32'] = transformer_attribution32.data.cpu().numpy()

RuntimeError: einsum(): subscript b has size 256 for operand 1 which does not broadcast with previously seen size 8

In [17]:
for block, data in data_dict_stt.items():
    # print(data['image'].unsqueeze(0).shape, data['met'].unsqueeze(0).shape, data['yz'].unsqueeze(0).shape)
    imgmet_attr, context_attr = attribution_generator.generate_LRP(data['image'].unsqueeze(0), 
                                                                 data['text'], 
                                                                 data['met'].unsqueeze(0), 
                                                                 data['yz'].unsqueeze(0),)
    
    print(imgmet_attr[0].shape, imgmet_attr[1].shape, context_attr.shape)
    # transformer_attribution = transformer_attribution.reshape(1, 4, 4, 4)
    # transformer_attribution4 = (transformer_attribution - transformer_attribution.min()) / (transformer_attribution.max() - transformer_attribution.min())
    # data_dict_stt[date_time]['lpr'] = transformer_attribution4.data.cpu().numpy()

ValueError: You need to specify either `text` or `text_target`.